### This workbook documents an attempt to leverage the Google News RSS feed to grab current and historical headlines on topics of interest.

The first step is to create a function to automatically create a query url with the custom search parameters.

In [1]:
!pip install --quiet torch
!pip install --quiet transformers

In [2]:
base_url = 'https://news.google.com/rss/search?q=us'
term_1 = 'stock'
term_2 = 'market'
term_3 = ''
day = '04'
month = '10'
year = '2024'
country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'

In [3]:
term_1_url = f'%20{term_1}'
term_2_url = f'%20{term_2}'
term_3_url = f'%20{term_3}'
month_url = f'%20when%3A{month}'
day_url = f'%2F{day}'
year_url = f'%2F{year}'

In [4]:
final_url = base_url + term_1_url + term_2_url + month_url + day_url + year_url + country_and_lang
final_url

'https://news.google.com/rss/search?q=us%20stock%20market%20when%3A10%2F04%2F2024&hl=en-US&gl=US&ceid=US%3Aen'

In [25]:
def create_url(term_1, term_2, term_3, term_4, term_5, day, month, year):
    base_url = 'https://news.google.com/rss/search?q=us'
    if term_1 != '':
        term_1_url = f'%20{term_1}'
    else:
        term_1_url = ''
    if term_2 != '':
        term_2_url = f'%20{term_2}'
    else:
        term_2_url = ''
    if term_3 != '':
        term_3_url = f'%20{term_3}'
    else:
        term_3_url = ''
    if term_4 != '':
        term_4_url = f'%20{term_4}'
    else:
        term_4_url = ''
    if term_5 != '':
        term_5_url = f'%20{term_4}'
    else:
        term_5_url = ''
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + term_1_url + term_2_url + term_3_url + term_4_url + term_5_url + month_url + day_url + year_url + country_and_lang
    return final_url

In [26]:
query_url = create_url('stock', 'market', 'outlook', '', '', '04', '10', '2024')
query_url

'https://news.google.com/rss/search?q=us%20stock%20market%20outlook%20when%3A10%2F04%2F2024&hl=en-US&gl=US&ceid=US%3Aen'

Next step is to use requests and BeautifulSoup to acquire and parse the xml from Google News' RSS feed. We will then search for all the headlines in the xml document.

In [7]:
import requests
from bs4 import BeautifulSoup

In [8]:
url = query_url
response = requests.get(url)
soup = BeautifulSoup(response.content, 'xml')
headlines = soup.find('channel').find_all('title')

In [9]:
hlist = []
headlines = soup.find('channel').find_all('title')
for item in headlines:
    hlist.append(item.get_text())
del hlist[0:2]
hlist

["Putting This Week's Events Into Context - J.P. Morgan",
 'Quarterly markets review - Q3 2024 - Schroders',
 'Jobs report stokes Wall Street rally that erases the week’s earlier losses - Fortune',
 'Higher share of electrified Porsche sports cars in the first quarter - Porsche Newsroom',
 'Could This Gold Stock Have Spectacular Gains? - Streetwise Reports',
 'Navigating the New Global Order: U.S. Foreign Policy in a Multipolar Era - TRENDS Research & Advisory',
 "JD Vance says 'illegal immigrants' are keeping you from owning your own home - Politico",
 'CRWD | CrowdStrike Holdings, Inc. Class Stock Data, Price & News - Quiver Quantitative',
 'Australian Alternatives sector shines amidst economic headwinds - Cushman & Wakefield',
 'JPMorgan warns Teleperformance stock could struggle to meet future expectations - Investing.com',
 'Michael Sprung’s Top Picks for October 4, 2024 - BNN Bloomberg',
 'Under new leadership, Alpargatas cuts costs and trims portfolio - Valor International',
 'H

We will now employ a BERT NLP model from huggingface trained specifically on a financial corpus to analyze the sentiment of the headlines.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')

In [11]:
source_model = f"ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(source_model)
model = AutoModelForSequenceClassification.from_pretrained(source_model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
def get_sentiment(example):
    encoded_text = tokenizer(example, return_tensors = 'pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        'finbert_pos' : scores[0],
        'finbert_neg' : scores[1],
        'finbert_neu' : scores[2]
    }
    return scores_dict

In [13]:
import pandas as pd
headlines_df = pd.DataFrame(hlist, columns = ['Headlines'])
hl_sentiment = headlines_df['Headlines'].apply(get_sentiment)

In [14]:
finbert_neg = []
for row in hl_sentiment:
    finbert_neg.append(list(row.values())[0])
finbert_neut = []
for row in hl_sentiment:
    finbert_neut.append(list(row.values())[1])
finbert_pos = []
for row in hl_sentiment:
    finbert_pos.append(list(row.values())[2])

In [15]:
final_hl_df = pd.DataFrame({
    'Headline': hlist,
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
})
final_hl_df

,Headline,Positive finBERT Score,Negative finBERT Score,Neutral finBERT Score
0,Putting This Week's Events Into Context - J.P....,0.046024,0.022782,0.931194
1,Quarterly markets review - Q3 2024 - Schroders,0.023042,0.047900,0.929058
2,Jobs report stokes Wall Street rally that eras...,0.142462,0.784796,0.072742
3,Higher share of electrified Porsche sports car...,0.817665,0.029046,0.153289
4,Could This Gold Stock Have Spectacular Gains? ...,0.583032,0.015018,0.401950
5,Navigating the New Global Order: U.S. Foreign ...,0.078442,0.018593,0.902965
6,JD Vance says 'illegal immigrants' are keeping...,0.021494,0.459940,0.518566
7,"CRWD | CrowdStrike Holdings, Inc. Class Stock ...",0.024309,0.037111,0.938580
8,Australian Alternatives sector shines amidst e...,0.379958,0.483083,0.136959
9,JPMorgan warns Teleperformance stock could str...,0.035176,0.579486,0.385338


In [16]:
avg_pos_sent = final_hl_df['Positive finBERT Score'].mean()
avg_neg_sent = final_hl_df['Negative finBERT Score'].mean()
avg_neut_sent = final_hl_df['Neutral finBERT Score'].mean()
print(f'Avg Pos Sentiment: {avg_pos_sent} | Avg Neg Sentiment: {avg_neg_sent} | Avg Neut Sentiment: {avg_neut_sent}')

Avg Pos Sentiment: 0.2222500592470169 | Avg Neg Sentiment: 0.18180213868618011 | Avg Neut Sentiment: 0.595947802066803


In [232]:
def create_url(term_1, term_2, term_3, term_4, term_5, day, month, year):
    base_url = 'https://news.google.com/rss/search?q=us'
    if term_1 != '':
        term_1_url = f'%20{term_1}'
    else:
        term_1_url = ''
    if term_2 != '':
        term_2_url = f'%20{term_2}'
    else:
        term_2_url = ''
    if term_3 != '':
        term_3_url = f'%20{term_3}'
    else:
        term_3_url = ''
    if term_4 != '':
        term_4_url = f'%20{term_4}'
    else:
        term_4_url = ''
    if term_5 != '':
        term_5_url = f'%20{term_4}'
    else:
        term_5_url = ''
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + term_1_url + term_2_url + term_3_url + term_4_url + term_5_url + month_url + day_url + year_url + country_and_lang
    return final_url

In [234]:
query_url = create_url('stock', 'market', '', '', '', '18', '04', '2026')
query_url

'https://news.google.com/rss/search?q=us%20stock%20market%20when%3A04%2F18%2F2026&hl=en-US&gl=US&ceid=US%3Aen'

In [235]:
def get_headlines(url):
    url = url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'xml')
    headlines = soup.find('channel').find_all('title')
    hlist = []
    dlist = []
    headlines = soup.find('channel').find_all('title')
    dates = soup.find('channel').find_all('pubDate')
    for item in headlines:
        hlist.append(item.get_text())
    for item in dates:
        dlist.append(item.get_text())
    del hlist[0:2]
    date_hl_df = pd.DataFrame({
    'Dates':dlist, 'Headlines':hlist})
    date_hl_df['Dates'] = pd.to_datetime(date_hl_df['Dates'])
    date_hl_df['Dates'] = date_hl_df['Dates'].dt.strftime('%d-%m-%y')
    date_hl_df = date_hl_df[date_hl_df['Dates'] == '18-04-26']
    return date_hl_df

In [236]:
get_headlines('https://news.google.com/rss/search?q=us%20stock%20market%20outlook%20when%3A04%2F18%2F2026&hl=en-US&gl=US&ceid=US%3Aen')

,Dates,Headlines
0,18-04-26,"Stock Market News Today — NYSE, NASDAQ & OTC H..."
1,18-04-26,‘Big sigh of relief’: Republicans finally get ...
2,18-04-26,Arizona Hits $1.60B in Early 2026 — Revenue Dr...
3,18-04-26,04/18/2026 6:45 AM Forecast - AOL.com
4,18-04-26,Identity as a Service (IDaaS) Market Outlook 2...
5,18-04-26,New Jersey Sports Betting Hits $1.01B in March...
10,18-04-26,"How Will Dow Jones Futures, Oil Prices React A..."


In [3]:
def load_model(source_model):
    source_model = source_model
    tokenizer = AutoTokenizer.from_pretrained(source_model)
    model = AutoModelForSequenceClassification.from_pretrained(source_model)
    return tokenizer, model

In [138]:
def get_sentiment(headlines_df):
    #define sentiment function
    def get_sentiment(example):
        encoded_text = tokenizer(example, return_tensors = 'pt')
        output = model(**encoded_text)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        scores_dict = {
            'finbert_pos' : scores[0],
            'finbert_neg' : scores[1],
            'finbert_neu' : scores[2]
        }
        return scores_dict
    
    #apply sentiment function
    hl_sentiment = headlines_df['Headlines'].apply(get_sentiment)
    finbert_neg = []
    for row in hl_sentiment:
        finbert_neg.append(list(row.values())[0])
    finbert_neut = []
    for row in hl_sentiment:
        finbert_neut.append(list(row.values())[1])
    finbert_pos = []
    for row in hl_sentiment:
        finbert_pos.append(list(row.values())[2])
    final_hl_df = pd.DataFrame({
    'Headline': headlines_df['Headlines'],
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
    })
    avg_pos = final_hl_df['Positive finBERT Score'].mean()
    avg_neg = final_hl_df['Negative finBERT Score'].mean()
    avg_neut = final_hl_df['Neutral finBERT Score'].mean()
    summary_table = pd.DataFrame({
    'Mean Pos Sentiment':avg_pos,
    'Mean Neg Sentiment':avg_neg,
    'Mean Neut Sentiment':avg_neut},
                                index = [0])
    return summary_table

In [139]:
get_sentiment(get_headlines('US Stock Market', '10-10-25'))

,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
0,0.098522,0.505008,0.396469


In [4]:
def get_sentiment(example):
    encoded_text = tokenizer(example, return_tensors = 'pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        'finbert_pos' : scores[0],
        'finbert_neg' : scores[1],
        'finbert_neu' : scores[2]
    }
    return scores_dict

In [134]:
def hl_sentiment_df(sentiment_function, headlines_df):
    hl_sentiment = headlines_df['Headlines'].apply(sentiment_function)
    finbert_neg = []
    for row in hl_sentiment:
        finbert_neg.append(list(row.values())[0])
    finbert_neut = []
    for row in hl_sentiment:
        finbert_neut.append(list(row.values())[1])
    finbert_pos = []
    for row in hl_sentiment:
        finbert_pos.append(list(row.values())[2])
    final_hl_df = pd.DataFrame({
    'Headline': headlines_df['Headlines'],
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
    })
    avg_pos = final_hl_df['Positive finBERT Score'].mean()
    avg_neg = final_hl_df['Negative finBERT Score'].mean()
    avg_neut = final_hl_df['Neutral finBERT Score'].mean()
    summary_table = pd.DataFrame({
    'Mean Pos Sentiment':avg_pos,
    'Mean Neg Sentiment':avg_neg,
    'Mean Neut Sentiment':avg_neut},
                                index = [0])
    return summary_table

In [1]:
!pip install --quiet torch transformers

import requests
from bs4 import BeautifulSoup
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
import warnings
warnings.filterwarnings('ignore')

In [11]:
tokenizer, model = load_model(f"ProsusAI/finbert")
hl_sentiment_df(get_sentiment, get_headlines(create_url('stock', 'market', 'outlook', '', '', '04', '10', '2024')))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
0,0.213785,0.141672,0.644543


testing

In [113]:
def get_headlines(url):
    url = url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'xml')
    headlines = soup.find('channel').find_all('title')
    hlist = []
    dlist = []
    headlines = soup.find('channel').find_all('title')
    dates = soup.find('channel').find_all('pubDate')
    for item in headlines:
        hlist.append(item.get_text())
    for item in dates:
        dlist.append(item.get_text())
    del hlist[0:2]
    date_hl_df = pd.DataFrame({
    'Dates':dlist, 'Headlines':hlist})
    date_hl_df['Dates'] = pd.to_datetime(date_hl_df['Dates'])
    date_hl_df['Dates'] = date_hl_df['Dates'].dt.strftime('%d-%m-%y')
    date_hl_df = date_hl_df[date_hl_df['Dates'] == '04-10-24']
    return date_hl_df

In [114]:
def create_url(query, date):
    base_url = 'https://news.google.com/rss/search?q=us'
    string_query = query
    split_string = string.split(' ')
    for word in split_string:
        base_url += f'%20{word}'
    date_str = (date).split('-')
    month = date_str[0]
    day = date_str[1]
    year = date_str[2]
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + month_url + day_url + year_url + country_and_lang
    return final_url

Streamlining url creation and requests query into one function

In [130]:
def get_headlines(query, date):
    #create url
    base_url = 'https://news.google.com/rss/search?q=us'
    string_query = query
    split_string = string.split(' ')
    for word in split_string:
        base_url += f'%20{word}'
    date_str = (date).split('-')
    month = date_str[0]
    day = date_str[1]
    year = date_str[2]
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + month_url + day_url + year_url + country_and_lang

    #conduct search with url and return dataframe with Date and Headline. Must filter for exact date.
    url = final_url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'xml')
    headlines = soup.find('channel').find_all('title')
    hlist = []
    dlist = []
    headlines = soup.find('channel').find_all('title')
    dates = soup.find('channel').find_all('pubDate')
    for item in headlines:
        hlist.append(item.get_text())
    for item in dates:
        dlist.append(item.get_text())
    del hlist[0:2]
    date_hl_df = pd.DataFrame({
    'Dates':dlist, 'Headlines':hlist})
    date_hl_df['Dates'] = pd.to_datetime(date_hl_df['Dates'])
    date_hl_df['Dates'] = date_hl_df['Dates'].dt.strftime('%d-%m-%y')
    date_hl_df = date_hl_df[date_hl_df['Dates'] == date]
    return date_hl_df

In [127]:
get_headlines('US Stock Market', '10-10-25')

,Dates,Headlines
2,10-10-25,'Liberation Day' in Retrospect: 6 Things That ...
3,10-10-25,Dow falls nearly 900 points after Trump reigni...
4,10-10-25,"Stock Market News Today, 10/10/25 – U.S. Stock..."
6,10-10-25,US Stocks Sink on Renewed China Tariff Threats...
7,10-10-25,Daily Natural Gas Market Update 10-10-25 - sto...
8,10-10-25,"Market Close Stock Round-Up October 10, 2025: ..."
9,10-10-25,Stocks hammered by tariff turmoil - Sherwood News
10,10-10-25,Dow plunges nearly 900 points after Trump blas...
16,10-10-25,US Stock Market Crash: Why is the US stock mar...
19,10-10-25,Energy Co. to Combine With Semiconductor Co. t...


### Finaled functions for loading model and getting headline sentiments for given dates and topics from Google RSS feed.

load_model function is kept seperate from headline_sentiment function for future utility in caching model info to avoid loading model each time a query is made

In [49]:
def load_model(source_model):
    source_model = source_model
    tokenizer = AutoTokenizer.from_pretrained(source_model)
    model = AutoModelForSequenceClassification.from_pretrained(source_model)
    return tokenizer, model

In [51]:
def headline_sentiment(query, date, tokenizer, model):
    #create url
    base_url = 'https://news.google.com/rss/search?q=us'
    string_query = query
    split_string = string_query.split(' ')
    for word in split_string:
        base_url += f'%20{word}'
    date_str = (date).split('-')
    month = date_str[0]
    day = date_str[1]
    year = date_str[2]
    month_url = f'%20when%3A{month}'
    day_url = f'%2F{day}'
    year_url = f'%2F{year}'
    country_and_lang = '&hl=en-US&gl=US&ceid=US%3Aen'
    final_url = base_url + month_url + day_url + year_url + country_and_lang

    #conduct search with url and return dataframe with Date and Headline. Must filter for exact date.
    url = final_url
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'xml')
    headlines = soup.find('channel').find_all('title')
    hlist = []
    dlist = []
    headlines = soup.find('channel').find_all('title')
    dates = soup.find('channel').find_all('pubDate')
    for item in headlines:
        hlist.append(item.get_text())
    for item in dates:
        dlist.append(item.get_text())
    del hlist[0:2]
    date_hl_df = pd.DataFrame({
    'Dates':dlist, 'Headlines':hlist})
    date_hl_df['Dates'] = pd.to_datetime(date_hl_df['Dates'])
    date_hl_df['Dates'] = date_hl_df['Dates'].dt.strftime('%m-%d-%Y')
    date_hl_df = date_hl_df[date_hl_df['Dates'] == date]
    
    #define sentiment function
    def get_sentiment(example):
        encoded_text = tokenizer(example, return_tensors = 'pt')
        output = model(**encoded_text)
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        scores_dict = {
            'finbert_pos' : scores[0],
            'finbert_neg' : scores[1],
            'finbert_neu' : scores[2]
        }
        return scores_dict
    
    #apply sentiment function and return mean sentiment of each type
    hl_sentiment = date_hl_df['Headlines'].apply(get_sentiment)
    finbert_neg = []
    for row in hl_sentiment:
        finbert_neg.append(list(row.values())[0])
    finbert_neut = []
    for row in hl_sentiment:
        finbert_neut.append(list(row.values())[1])
    finbert_pos = []
    for row in hl_sentiment:
        finbert_pos.append(list(row.values())[2])
    final_hl_df = pd.DataFrame({
    'Headline': date_hl_df['Headlines'],
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
    })
    avg_pos = final_hl_df['Positive finBERT Score'].mean()
    avg_neg = final_hl_df['Negative finBERT Score'].mean()
    avg_neut = final_hl_df['Neutral finBERT Score'].mean()
    summary_table = pd.DataFrame({
    'Date':date,
    'Mean Pos Sentiment':avg_pos,
    'Mean Neg Sentiment':avg_neg,
    'Mean Neut Sentiment':avg_neut},
                                index = [0])
    return summary_table

In [52]:
tokenizer, model = load_model('ProsusAI/finbert')
headline_sentiment('US Economy', '04-17-2026', tokenizer, model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Date,Mean Pos Sentiment,Mean Neg Sentiment,Mean Neut Sentiment
0,04-17-2026,0.040975,0.135036,0.823989
